# 06. Modeling
**목적**: label 유무에 따라 파이프라인을 분기한다.

- **Case A (label 있음)**: train_features.parquet 직접 사용 → LightGBM supervised 분류 모델
- **Case B (label 없음)**: Risk Index 기반 우선순위화 + HDBSCAN 클러스터링

두 경우 모두 07_explainability 에서 사용할 결과물을 저장한다.

In [ ]:
import sys
sys.path.append('..')

import json
import pickle
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from config import DATA_PROCESSED, OUT_TABLES, SEED

with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)
FLAGS  = cmap['flags']
FC     = cmap['facility']
FID_COL = FC['facility_id']

print(f'모델 유형: {"Case A" if FLAGS["has_label"] else "Case B"}')

---
## Case A — Supervised ML (label 있을 때)

train_features.parquet 에 날짜·공간 기반 label과 모든 feature가 포함되어 있음.

In [ ]:
if FLAGS['has_label']:
    from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score
    import lightgbm as lgb
    import xgboost as xgb

    # train_features.parquet 로드 (label + 모든 feature)
    tf = pd.read_parquet(DATA_PROCESSED / 'train_features.parquet')
    tf['date'] = pd.to_datetime(tf['date'])

    LABEL_COL = 'label'
    EXCLUDE   = {FID_COL, 'date', LABEL_COL, 'nearest_station'}
    FEATURE_COLS = [c for c in tf.columns
                    if c not in EXCLUDE
                    and tf[c].dtype in [np.float64, np.float32, np.int64, np.int32, np.int8]]

    n_pos = tf[LABEL_COL].sum()
    n_neg = (tf[LABEL_COL] == 0).sum()
    spw   = int(n_neg / max(n_pos, 1))

    print(f'데이터: {tf.shape}  |  feature 수: {len(FEATURE_COLS)}')
    print(f'label=0: {n_neg:,}  label=1: {n_pos:,}  scale_pos_weight: {spw}')
    print(f'feature 예시: {FEATURE_COLS[:8]}')

In [ ]:
if FLAGS['has_label']:
    # Time-based split: 최근 20% → test
    dates_sorted = tf['date'].sort_values().unique()
    cutoff = dates_sorted[int(len(dates_sorted) * 0.8)]

    train_mask = tf['date'] < cutoff
    test_mask  = tf['date'] >= cutoff

    X = tf[FEATURE_COLS].fillna(tf[FEATURE_COLS].median())
    y = tf[LABEL_COL].astype(int)

    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
    print(f'Test 기간: {tf.loc[test_mask, "date"].min().date()} ~ {tf.loc[test_mask, "date"].max().date()}')
    print(f'Test label=1: {y_test.sum():,} / {len(y_test):,} ({y_test.mean()*100:.2f}%)')

In [ ]:
if FLAGS['has_label']:
    results = {}

    models = {
        'lightgbm': lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.05, num_leaves=63,
            random_state=SEED, scale_pos_weight=spw, verbose=-1),
        'xgboost': xgb.XGBClassifier(
            n_estimators=500, learning_rate=0.05, random_state=SEED,
            eval_metric='logloss', scale_pos_weight=spw, verbosity=0),
    }

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        # Recall@Top-K (상위 10% 예측 내 실제 화재 비율)
        k10 = max(1, int(len(y_test) * 0.10))
        top_k_idx = np.argsort(y_prob)[::-1][:k10]
        recall_topk = y_test.values[top_k_idx].sum() / max(1, y_test.sum())

        results[name] = {
            'AUC':            roc_auc_score(y_test, y_prob),
            'F1':             f1_score(y_test, y_pred, zero_division=0),
            'Recall':         recall_score(y_test, y_pred, zero_division=0),
            'Precision':      precision_score(y_test, y_pred, zero_division=0),
            'Recall@Top10%':  recall_topk,
        }
        print(f'{name}: AUC={results[name]["AUC"]:.3f}  F1={results[name]["F1"]:.3f}  Recall@Top10%={recall_topk:.3f}')

    df_results = pd.DataFrame(results).T
    print('\n=== 모델 성능 비교 ===')
    print(df_results.round(3))
    OUT_TABLES.mkdir(parents=True, exist_ok=True)
    df_results.to_csv(OUT_TABLES / 'model_performance.csv')

    # 최종 모델 선택 (AUC 기준)
    best_name = df_results['AUC'].idxmax()
    best_model = models[best_name]
    print(f'\n최종 모델: {best_name}')

    with open(DATA_PROCESSED / 'best_model.pkl', 'wb') as f:
        pickle.dump({'model': best_model, 'feature_cols': FEATURE_COLS, 'name': best_name}, f)
    print('모델 저장 완료: best_model.pkl')

    # LightGBM 별도 저장 (SHAP용)
    with open(DATA_PROCESSED / 'lgbm_model.pkl', 'wb') as f:
        pickle.dump({'model': models['lightgbm'], 'feature_cols': FEATURE_COLS}, f)
    print('LightGBM 저장 완료: lgbm_model.pkl')

---
## Case B — Risk Prioritisation + Clustering (label 없을 때)

In [ ]:
if not FLAGS['has_label']:
    from sklearn.preprocessing import MinMaxScaler
    print('Case B: Rule-based Risk Index + Spatial Clustering')

    df = pd.read_parquet(DATA_PROCESSED / 'risk_scores.parquet')
    FID_COL = FC['facility_id']

    latest = df.groupby(FID_COL).agg(
        final_risk_mean=('final_risk', 'mean'),
        final_risk_max=('final_risk', 'max'),
        final_risk_last=('final_risk', 'last'),
        high_risk_days=('risk_grade', lambda x: (x.isin(['High', 'Very High'])).sum()),
    ).reset_index()

    scaler = MinMaxScaler(feature_range=(0, 100))
    latest['priority_score'] = (
        0.60 * latest['final_risk_last'] +
        0.25 * latest['final_risk_max'] +
        0.15 * scaler.fit_transform(latest[['high_risk_days']]).flatten()
    ).clip(0, 100)

    latest = latest.sort_values('priority_score', ascending=False).reset_index(drop=True)
    latest['inspection_rank'] = latest.index + 1

    print(f'설비 수: {len(latest)}')
    print(latest.head(10))

    latest.to_csv(OUT_TABLES / 'inspection_priority.csv', index=False)
    print('inspection_priority.csv 저장 완료')

    with open(DATA_PROCESSED / 'case_b_result.pkl', 'wb') as f:
        pickle.dump({'priority_df': latest}, f)
    print('case_b_result.pkl 저장 완료')

print('\n다음 단계: 07_explainability.ipynb')